# Demo extra: el mismo agente en una tienda real (soyHouston256/store)

**Opcional. No forma parte de la evaluación oficial**, que está en `Laboratorio4_Asistente_Compras_Voz.ipynb`
(ESCI + tienda simulada).

Aquí el *mismo* agente, sin cambiar su lógica, compra en una tienda real de merch (polos, tazas y mousepads)
a través de `shopassist.remote_store.RemoteStoreAdapter`, que tiene la misma interfaz que `Store`:

* catálogo y precios (PEN) desde `GET {STORE_API_URL}/api/products`, indexados con BM25;
* la API no tiene stock numérico: el stock se **simula** en una copia SQLite reproducible (`soldOut` → 0);
* antes de confirmar se vuelve a leer cada producto (`GET /api/products/:id`) para detectar cambios de precio o agotados;
* el agente pide **talla / color / corte** cuando faltan.

Sin `STORE_API_URL` se usa una copia offline del catálogo (`scenarios/tienda_real_fixture.json`).

> ⚠️ **Pedidos reales.** Por defecto todo es **DRY-RUN**: el pedido se registra solo en local y se muestra el *payload*
> que se enviaría a `POST /api/orders`. Un pedido **real** solo se crea con `allow_real_orders=True` **y**
> `customer={"dni", "name", "phone"}`. Nunca se hace POST dos veces para la misma cotización.

In [ ]:
# Trae el código a Colab (elige UNA opción)
import os, sys
REPO_URL = ""          # opción 1: URL del repo, p.ej. "https://github.com/<usuario>/purchase.git"
ZIP_IN_DRIVE = "/content/drive/MyDrive/purchase.zip"   # opción 2: zip en tu Drive
WORKDIR = "/content/purchase"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists(WORKDIR):
    if REPO_URL:
        !git clone -q {REPO_URL} {WORKDIR}
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        !unzip -q -o {ZIP_IN_DRIVE} -d /content/
if os.path.exists(WORKDIR):
    os.chdir(WORKDIR)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())
# Recupera grabaciones guardadas antes en Drive (opcional)
if IN_COLAB and os.path.exists("/content/drive/MyDrive/purchase/voice"):
    !cp -rn /content/drive/MyDrive/purchase/voice/* voice/

In [ ]:
# Instala las dependencias (versiones fijadas en requirements.txt)
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null

In [ ]:
import os, json
import pandas as pd
from IPython.display import display
from shopassist.config import Config
from shopassist.remote_store import build_remote_demo

cfg = Config()
os.environ.setdefault("STORE_API_URL", "https://api.devhaus.pe")   # vacío = copia offline
demo = build_remote_demo(cfg)                # dry-run (allow_real_orders=False por defecto)
tienda, agent_factory_tienda = demo["store"], demo["agent_factory"]
print("Modo:", "OFFLINE (fixture)" if demo["offline"] else f"API {tienda.base_url}", "| productos:", len(tienda.product_ids()),
      "| moneda:", demo["cfg"].currency)
display(tienda.catalog_df[["product_id", "title", "color", "bullet_point"]].head(8))

In [ ]:
tienda.restore(cfg.SEED)
agente_tienda = agent_factory_tienda("demo_tienda")
for u in ["Quiero un polo", "El de Redis", "Talla M, color negro",
          "Quiero también una taza de React blanca", "Sí, confirmo el pedido", "Sí, confirmo el pedido"]:
    t = agente_tienda.handle(u)
    print(f"👤 {u}\n🤖 {t.reply}\n")
for o in tienda.list_orders():
    print(o["order_id"], o["total"], o["remote"]["mode"], "→", o["remote"]["endpoint"])
    print(json.dumps(o["remote"]["payload"], ensure_ascii=False, indent=1))

In [ ]:
# Escenarios de la tienda real (dry-run): ambigüedad, talla/color, agotado, taza
from shopassist.scenarios import load_scenarios, run_all
df_tienda, res_tienda = run_all(load_scenarios("scenarios/scenarios_tienda_real.yaml"), agent_factory_tienda, tienda,
                                out_dir="results/tienda_real")
display(df_tienda[["id", "titulo", "ok", "checks_ok", "checks", "fallidos"]])

In [ ]:
# (NO ejecutar salvo que se quiera crear un pedido REAL en la tienda)
# demo_real = build_remote_demo(cfg, allow_real_orders=True,
#                               customer={"dni": "...", "name": "...", "phone": "..."})